# Extracting data from website

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/131.0.0.0 Safari/537.36"
}

results = []

for jobs in job_url:  # job_url is your list of posting URLs
    response = requests.get(jobs, headers=headers, timeout=10)
    if response.status_code != 200:
        # print(f"Skipped {jobs}")
        continue

    soup = BeautifulSoup(response.content, "html.parser")

    # --- Job Title ---
    job_title_tag = soup.find("h1")
    job_title = job_title_tag.get_text(strip=True) if job_title_tag else None

    # --- Employer ---
    employer = None
    employer_strong = soup.find("strong", string=lambda x: x and "Employer:" in x)
    if employer_strong:
        employer = employer_strong.parent.get_text(strip=True).replace("Employer:", "").strip()

    # --- Employer URL ---
    employer_url = None
    employer_url_strong = soup.find("strong", string=lambda x: x and "Employer URL:" in x)
    if employer_url_strong:
        link = employer_url_strong.find_next("a")
        employer_url = link["href"] if link else None

    # --- Posted Date ---
    posted_date = None
    posted_strong = soup.find("strong", string=lambda x: x and "Posted to IASSIST on:" in x)
    if posted_strong:
        posted_date = posted_strong.parent.get_text(strip=True).replace("Posted to IASSIST on:", "").strip()

    # --- Salary ---
    salary = None
# look for either "Salary" or "Benefits"
    comp_strong = soup.find("strong", string=lambda x: x and ("Salary" in x or "Benefits" in x))
    if comp_strong:
      next_p = comp_strong.parent.find_next_sibling("p")
      salary = next_p.get_text(strip=True) if next_p else None


    # --- Necessary Skills ---
    skills_text = None
    skills = []
    skills_label = soup.find("p", string=lambda x: x and "Necessary Skills:" in x)
    if skills_label:
        ul = skills_label.find_next_sibling("ul")
        if ul:
            for li in ul.find_all("li"):
                skills.append(li.get_text(strip=True))
    skills_text = "; ".join(skills) if skills else None

        # --- Job Location ---
    job_location = None
    location_strong = soup.find("strong", string=lambda x: x and "Job Location" in x)
    if location_strong:
        next_p = location_strong.find_parent().find_next_sibling("p")
        job_location = next_p.get_text(strip=True) if next_p else None


    # --- Preferred Skills ---
    preferred_text = None
    preferred = []
    preferred_label = soup.find("p", string=lambda x: x and "Preferred Skills:" in x)
    if preferred_label:
        ul = preferred_label.find_next_sibling("ul")
        if ul:
            for li in ul.find_all("li"):
                preferred.append(li.get_text(strip=True))
    preferred_text = "; ".join(preferred) if preferred else None

    edu_text = None
    edu = []

# Look for all <strong> tags that mention Required or Preferred Qualifications
    qual_headers = soup.find_all("strong", string=lambda x: x and ("Qualifications" in x))

    for header in qual_headers:
    # Get the next <ul> sibling after the header (may be wrapped in a <p>)
      next_ul = header.find_parent().find_next_sibling("ul")
      if next_ul:
        for li in next_ul.find_all("li"):
            edu.append(li.get_text(strip=True))

      edu_text = "; ".join(edu) if edu else None



    # Store results
    results.append({
        "Job Title": job_title,
        "Employer": employer,
        "Employer URL": employer_url,
        "Posted Date": posted_date,
        "Salary": salary,
        "Job Location":job_location,
        "Necessary Skills": skills_text,
        "Preferred Skills": preferred_text,
        "Education/Experience": edu_text,
        "Source URL": jobs
    })


In [ ]:
df = pd.DataFrame(results)

# Data Cleaning and Analysis


In [ ]:
df

In [ ]:
df['Posted Date'] = pd.to_datetime(df['Posted Date'], errors='coerce')

# Define the date range
start_date = '2010-01-01'
end_date = '2025-12-31'

# Filter rows where 'Date' is between start_date and end_date (inclusive)
filtered_df = df[(df['Posted Date'] >= start_date) & (df['Posted Date'] <= end_date)]

In [ ]:
filtered_df.info()

In [ ]:
filtered_df

In [ ]:
pattern = r'research data librarian|data librarian|data managers|data curation librarian'

# Filter rows where 'Jo b Title' contains any of the keywords
filtered_jobs = filtered_df[filtered_df['Job Title'].str.contains(pattern, case=False, na=False)]

In [ ]:
filtered_jobs.info()

In [ ]:
filtered_jobs =filtered_jobs.drop(columns=["Necessary Skills", "Preferred Skills"])


In [ ]:
subset_df = filtered_jobs[["Education/Experience", "Salary", "Source URL", "Job Location"]].dropna(how="any")
subset_df = filtered_jobs[filtered_jobs['Education/Experience'].notnull()]

# Select the desired columns
result_df = subset_df[['Salary', 'Source URL', 'Job Location', 'Education/Experience']]

# Reset index if you want a clean DataFrame
result_df = result_df.reset_index(drop=True)

In [ ]:
result_df

In [ ]:
# 1. Define the update mapping and drop list
update_map = {
    "https://iassistdata.org/jobs-repository/2023-12-18/": {"Salary": "$74,280 - $77,508", "Job Location": None},
    "https://iassistdata.org/jobs-repository/2023-10-06/": {"Salary": "66,667-78,000", "Job Location": None},
    "https://iassistdata.org/jobs-repository/2024-07-24_1/": {"Salary": "$65,000-$70,000", "Job Location": None},
    "https://iassistdata.org/jobs-repository/2022-04-19-00749/": {"Salary": None, "Job Location": "FA01 - Faculty - South Orange, NJ"},
    "https://iassistdata.org/jobs-repository/2022-03-03-00732/": {"Salary": "$69,100 - $116,100", "Job Location": "Cambridge, MA"},
    "https://iassistdata.org/jobs-repository/2021-10-18-00705/": {"Salary": "$55,000-$85,000", "Job Location": "NY"},
    "https://iassistdata.org/jobs-repository/2021-04-16-00695-1/": {"Salary": "$72,367 - $92,613", "Job Location": "Toronto, CA"},
    "https://iassistdata.org/jobs-repository/2021-03-02-00687/": {"Salary": "$65,000 - $75,000", "Job Location": "Columbus, OH"},
    "https://iassistdata.org/jobs-repository/2020-09-17-00653/": {"Salary": "$48,000", "Job Location": "Logan, UT"},
    "https://iassistdata.org/jobs-repository/2019-07-08-00611/": {"Salary": "$48,000", "Job Location": "Ithaca, NY"},
    "https://iassistdata.org/jobs-repository/2018-12-13-00568/": {"Salary": "$76,603", "Job Location": "Ann Arbor, MI"},
    "https://iassistdata.org/jobs-repository/2018-10-30-00558/": {"Salary": "$47,428", "Job Location": "Knoxville, TN"},
    "https://iassistdata.org/jobs-repository/2017-11-29-00513/": {"Salary": None, "Job Location": "Edinburgh, UK"},
    "https://iassistdata.org/jobs-repository/2016-09-22-00428/": {"Salary": None, "Job Location": "College Station, TX"},
    "https://iassistdata.org/jobs-repository/2014-11-17-00286/": {"Salary": "$76,603", "Job Location": "Ann Arbor, MI"},
    "https://iassistdata.org/jobs-repository/2025-02-10/": {"Salary": "$65,226 - $86,816", "Job Location": None},
    "https://iassistdata.org/jobs-repository/2024-12-12/": {"Salary": "$60,000", "Job Location": None},
    "https://iassistdata.org/jobs-repository/2024-07-25/": {"Salary": "$63,000", "Job Location": None},
    "https://iassistdata.org/jobs-repository/2025-03-19/": {"Salary": "$76,403 - $79,720", "Job Location": None}
}

drop_urls = [
    "https://iassistdata.org/jobs-repository/2017-05-18-00471/",
    "https://iassistdata.org/jobs-repository/2016-11-16-00444/",
    "https://iassistdata.org/jobs-repository/2012-04-05-00171/",
]

# 2. Update the original DataFrame in place
for url, updates in update_map.items():
    mask = filtered_df['Source URL'] == url
    if updates["Salary"] is not None:
        filtered_df.loc[mask, "Salary"] = updates["Salary"]
    if updates["Job Location"] is not None:
        filtered_df.loc[mask, "Job Location"] = updates["Job Location"]

# 3. Filter to rows with non-null Education/Experience
filtered_result_df = filtered_df[filtered_df['Education/Experience'].notnull()].copy()

# 4. Drop rows with specified Source URLs
filtered_result_df = filtered_result_df[~filtered_result_df['Source URL'].isin(drop_urls)]

# 5. Reset index if needed
filtered_result_df = filtered_result_df.reset_index(drop=True)


In [ ]:
filtered_result_df

In [ ]:
filtered_result_df.info()

In [ ]:
# Filter rows where 'Job Location' is not null
filtered_by_location = filtered_df[filtered_df['Job Location'].notnull()]

# Optional: reset index for clean DataFrame
filtered_by_location = filtered_by_location.reset_index(drop=True)


In [ ]:
filtered_by_location = filtered_by_location.drop(columns=["Necessary Skills", "Preferred Skills"])


In [ ]:
filtered_by_location.info()

In [ ]:
# filtered_by_location.head(25)
pattern = r'research data librarian|data librarian|data managers|data curation librarian'

# Filter rows where 'Job Title' contains any of the keywords
iassist_jobs = filtered_by_location[filtered_by_location['Job Title'].str.contains(pattern, case=False, na=False)]

In [ ]:
# Fill missing values in 'Salary' and 'Education/Experience' columns with 'not provided'
iassist_jobs['Salary'] = iassist_jobs['Salary'].fillna('not provided')
iassist_jobs['Education/Experience'] = iassist_jobs['Education/Experience'].fillna('not provided')

In [ ]:
iassist_jobs.info()

In [ ]:
import pandas as pd

# Function to extract only the "skills" part
def extract_skills(text):
    if pd.isna(text):
        return None

    parts = text.split(";", 1)  # split into two parts only
    if len(parts) > 1:
        return parts[1].strip()  # everything after the first ";"
    return None  # no skills if no ";"

# Apply function to create new column
iassist_jobs["Skills"] = iassist_jobs["Education/Experience"].apply(extract_skills)


In [ ]:
iassist_jobs

In [ ]:
iassist_jobs[["Salary"]]

In [ ]:
iassist_jobs.to_csv("iassist_jobs.csv", index=False)

# GRAPHS

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import re

# Example: iassist_jobs is your DataFrame

def extract_min_salary(s):
    if pd.isnull(s):
        return None
    s = s.lower()
    # Treat common non-numeric entries as missing
    if 'not provided' in s or 'relocation' in s or 'faculty' in s:
        return None
    # Extract all numbers (digits) ignoring commas and dollar signs
    nums = re.findall(r'\$?(\d{2,6})', s.replace(',', ''))
    if nums:
        # Return the first number as minimum salary
        return int(nums[0])
    return None

# Apply extraction to Salary column
iassist_jobs['Min Salary'] = iassist_jobs['Salary'].apply(extract_min_salary)

# Drop rows where Min Salary is missing for plotting
salary_data = iassist_jobs['Min Salary'].dropna().astype(int)

# Plot histogram of minimum salaries
plt.figure(figsize=(8,4))
plt.hist(salary_data, bins=10, color='skyblue', edgecolor='black')
plt.title('Distribution of Minimum Salaries')
plt.xlabel('Minimum Salary ($)')
plt.ylabel('Number of Jobs')
plt.tight_layout()
plt.show()


In [ ]:
# Extract state abbreviation from 'Job Location' (format: "City, ST")
iassist_jobs['State'] = iassist_jobs['Job Location'].apply(
    lambda x: x.split(',')[-1].strip() if pd.notnull(x) and ',' in x else 'Unknown'
)

# Plot 1: Job postings by state
state_counts = iassist_jobs['State'].value_counts()
plt.figure(figsize=(10,5))
state_counts.plot(kind='bar', title='Job Postings by State')
plt.xlabel('State')
plt.ylabel('Number of Jobs')
plt.tight_layout()
plt.show()


In [ ]:
# Plot 2: Top 10 Employers by number of job postings
top_employers = iassist_jobs['Employer'].value_counts().head(10)
plt.figure(figsize=(8,4))
top_employers.plot(kind='bar', title='Top 10 Employers')
plt.xlabel('Employer')
plt.ylabel('Number of Jobs')
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

top_titles = iassist_jobs['Job Title'].value_counts().head(10)

plt.figure(figsize=(10, 6))
top_titles.plot(kind='barh', color='orange')
plt.title("Top 10 Job Titles")
plt.xlabel("Number of Postings")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Group by 'State' and calculate average minimum salary
avg_salary_by_state = iassist_jobs.groupby('State')['Min Salary'].mean().dropna()

# Plot average salary by state as a bar chart
plt.figure(figsize=(10,5))
avg_salary_by_state.sort_values(ascending=False).plot(kind='bar', color='skyblue')
plt.title('Average Minimum Salary by State')
plt.xlabel('State')
plt.ylabel('Average Minimum Salary ($)')
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Example: your DataFrame is iassist_jobs and it has a column 'Education/Experience'

# Step 1: Define a function to extract and categorize education/experience levels
def extract_edu_exp(text):
    if pd.isna(text):
        return 'Not Provided'
    text = text.lower()
    if 'phd' in text or 'doctorate' in text:
        return 'PhD/Doctorate'
    elif 'master' in text or 'msc' in text or 'ma ' in text or 'm.a.' in text or 'm.s.' in text:
        return 'Master’s Degree'
    elif 'bachelor' in text or 'ba ' in text or 'b.a.' in text or 'b.s.' in text or 'bs ' in text:
        return 'Bachelor’s Degree'
    elif 'associate' in text:
        return 'Associate Degree'
    elif 'experience' in text:
        return 'Experience Required'
    else:
        return 'Other/Unspecified'

# Step 2: Apply the function to create a new categorized column
iassist_jobs['EduExp_Category'] = iassist_jobs['Education/Experience'].apply(extract_edu_exp)

# Step 3: Plot the histogram (bar chart) of counts per category
plt.figure(figsize=(8,4))
iassist_jobs['EduExp_Category'].value_counts().plot(
    kind='bar', color='skyblue', edgecolor='black'
)
plt.title('Histogram of Education/Experience Categories')
plt.xlabel('Education/Experience Category')
plt.ylabel('Number of Job Postings')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


# Visual Data Succesfully extracted


# Rough Draft of various attempts to get data based on the goal of the project


In [ ]:
filtered_jobs.to_csv("./iassist_jobs.csv")

In [ ]:
iassist_jobs.info()

In [ ]:



# 1. Define the update mapping and drop list
update_map = {
    "https://iassistdata.org/jobs-repository/2023-12-18/": {"Salary": "$74,280 - $77,508", "Job Location": None},
    "https://iassistdata.org/jobs-repository/2023-10-06/": {"Salary": "66,667-78,000", "Job Location": None},
    "https://iassistdata.org/jobs-repository/2024-07-24_1/": {"Salary": "$65,000-$70,000", "Job Location": None},
    "https://iassistdata.org/jobs-repository/2022-04-19-00749/": {"Salary": None, "Job Location": "FA01 - Faculty - South Orange, NJ"},
    "https://iassistdata.org/jobs-repository/2022-03-03-00732/": {"Salary": "$69,100 - $116,100", "Job Location": "Cambridge, MA"},
    "https://iassistdata.org/jobs-repository/2021-10-18-00705/": {"Salary": "$55,000-$85,000", "Job Location": "NY"},
    "https://iassistdata.org/jobs-repository/2021-04-16-00695-1/": {"Salary": "$72,367 - $92,613", "Job Location": "Toronto, CA"},
    "https://iassistdata.org/jobs-repository/2021-03-02-00687/": {"Salary": "$65,000 - $75,000", "Job Location": "Columbus, OH"},
    "https://iassistdata.org/jobs-repository/2020-09-17-00653/": {"Salary": "$48,000", "Job Location": "Logan, UT"},
    "https://iassistdata.org/jobs-repository/2019-07-08-00611/": {"Salary": "$48,000", "Job Location": "Ithaca, NY"},
    "https://iassistdata.org/jobs-repository/2018-12-13-00568/": {"Salary": "$76,603", "Job Location": "Ann Arbor, MI"},
    "https://iassistdata.org/jobs-repository/2018-10-30-00558/": {"Salary": "$47,428", "Job Location": "Knoxville, TN"},
    "https://iassistdata.org/jobs-repository/2017-11-29-00513/": {"Salary": None, "Job Location": "Edinburgh, UK"},
    "https://iassistdata.org/jobs-repository/2016-09-22-00428/": {"Salary": None, "Job Location": "College Station, TX"},
    "https://iassistdata.org/jobs-repository/2014-11-17-00286/": {"Salary": "$76,603", "Job Location": "Ann Arbor, MI"},
    "https://iassistdata.org/jobs-repository/2025-02-10/": {"Salary": "$65,226 - $86,816", "Job Location": None},
    "https://iassistdata.org/jobs-repository/2024-12-12/": {"Salary": "$60,000", "Job Location": None},
     "https://iassistdata.org/jobs-repository/2024-07-25/": {"Salary": "$63,000", "Job Location": None},
     "https://iassistdata.org/jobs-repository/2025-03-19/": {"Salary": "$76,403 - $79,720", "Job Location": None}


}

drop_urls = [
    "https://iassistdata.org/jobs-repository/2017-05-18-00471/",
    "https://iassistdata.org/jobs-repository/2016-11-16-00444/",
    "https://iassistdata.org/jobs-repository/2012-04-05-00171/",
]

# 2. Filter to rows with non-null Education/Experience
filtered_result_df = result_df[result_df['Education/Experience'].notnull()].copy()

# 3. Drop rows with specified Source URLs
filtered_result_df = filtered_result_df[~filtered_result_df['Source URL'].isin(drop_urls)]

# 4. Update Salary and Job Location based on Source URL
for url, updates in update_map.items():
    mask = filtered_result_df['Source URL'] == url
    if updates["Salary"] is not None:
        filtered_result_df.loc[mask, "Salary"] = updates["Salary"]
    if updates["Job Location"] is not None:
        filtered_result_df.loc[mask, "Job Location"] = updates["Job Location"]

# 5. Reset index and select relevant columns
final_df = filtered_result_df.reset_index(drop=True)[['Salary', 'Source URL', 'Job Location', 'Education/Experience']]



In [ ]:
final_df

In [ ]:
# Filter original DataFrame rows where 'Source URL' is in final_df's 'Source URL' column
result_full = filtered_jobs[filtered_jobs['Source URL'].isin(final_df['Source URL'])]


In [ ]:
result_full

In [ ]:
result_full.info()

In [ ]:
filtered_sub.head(25)

In [ ]:
filtered_jobs.head(25)

In [ ]:
import matplotlib.pyplot as plt

top_titles = filtered_jobs['Job Title'].value_counts().head(10)

plt.figure(figsize=(10, 6))
top_titles.plot(kind='barh', color='orange')
plt.title("Top 10 Job Titles")
plt.xlabel("Number of Postings")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
if 'Job Location' in filtered_jobs.columns:
    top_locations = filtered_jobs['Job Location'].dropna().value_counts().head(10)

    plt.figure(figsize=(10, 6))
    top_locations.plot(kind='barh', color='green')
    plt.title("Top 10 Job Locations")
    plt.xlabel("Number of Postings")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()


In [ ]:
filtered_jobs.to_csv("./iassist_jobs.csv")

In [ ]:
import pandas as pd

In [ ]:
data = pd.read_csv("./data_jobs.csv")

In [ ]:
data

In [ ]:
data.info()

In [ ]:
cols_to_drop = [
    "companyInfo/companyLogo",
    "companyInfo/companySize",
    "companyInfo/companySize/max",
    "companyInfo/reviewCount",
    "rating",
    "reviewsCount",
    "companyInfo/companySize/min",
    "companyInfo/indeedUrl",
    "companyInfo/companyDescription",
    "jobType/1",
    "companyInfo/url"
]

data = data.drop(columns=cols_to_drop)

In [ ]:
data.info()

In [ ]:
data = data.rename(columns={
    "positionName": "Job Title",
    "salary": "Salary",
    "jobType/0": "Employment Type",
    "company": "Company Name",
    "location": "Job Location",
    "url": "Job URL",
    "companyInfo/rating": "Company Rating"
})

In [ ]:
data.head()

In [ ]:
pattern = "librarian"
filtered_jobs = data[data['Job Title'].str.contains(pattern, case=False, na=False)]

In [ ]:
final_data = data[data["Job Title"].str.contains(r"(librarian|research)", case=False, na=False)]

In [ ]:
final_data

In [ ]:
final_data.to_csv("./final_data.csv", index=False)

In [ ]:
filtered_jobs.info()

In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

url = "https://iassistdata.org/jobs-repository/"
headers = {
    "User-Agent": "Mozilla/5.0 (Linux; Android 6.0; Nexus 5 Build/MRA58N) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Mobile Safari/537.36"
}

web_data = requests.get(url, headers=headers)
soup = BeautifulSoup(web_data.content, features="html.parser")

# Find all job posting list items
li_items = soup.find_all('li', class_='alternate')
job_url = []
# For each job, extract the link to the detail page
for li in li_items:
    a_tag = li.find('a')
    if a_tag and 'href' in a_tag.attrs:
        job_title = a_tag.get_text(strip=True)
        job_url.append(a_tag['href'])


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/131.0.0.0 Safari/537.36"
}

results = []

# Define qualification keyword pattern
QUAL_PATTERN = r"(master|phd|doctorate|degree|ala|professor|instructor|bachelor)"

def split_edu_experience(text):
    """Split Education/Experience text into Qualification and Skills"""
    if not text:
        return None, None

    parts = [p.strip() for p in text.split(";") if p.strip()]

    qualifications = []
    skills = []

    for p in parts:
        if re.search(QUAL_PATTERN, p, re.IGNORECASE):
            qualifications.append(p)
        else:
            skills.append(p)

    qualification_text = "; ".join(qualifications) if qualifications else None
    skills_text = "; ".join(skills) if skills else None

    return qualification_text, skills_text


for jobs in job_url:  # job_url is your list of posting URLs
    response = requests.get(jobs, headers=headers, timeout=10)
    if response.status_code != 200:
        print(f"❌ Skipped {jobs}")
        continue

    soup = BeautifulSoup(response.content, "html.parser")

    # --- Job Title ---
    job_title_tag = soup.find("h1")
    job_title = job_title_tag.get_text(strip=True) if job_title_tag else None

    # --- Employer ---
    employer = None
    employer_strong = soup.find("strong", string=lambda x: x and "Employer:" in x)
    if employer_strong:
        employer = employer_strong.parent.get_text(strip=True).replace("Employer:", "").strip()

    # --- Employer URL ---
    employer_url = None
    employer_url_strong = soup.find("strong", string=lambda x: x and "Employer URL:" in x)
    if employer_url_strong:
        link = employer_url_strong.find_next("a")
        employer_url = link["href"] if link else None

    # --- Posted Date ---
    posted_date = None
    posted_strong = soup.find("strong", string=lambda x: x and "Posted to IASSIST on:" in x)
    if posted_strong:
        posted_date = posted_strong.parent.get_text(strip=True).replace("Posted to IASSIST on:", "").strip()

    # --- Salary / Benefits ---
    salary = None
    comp_strong = soup.find("strong", string=lambda x: x and ("Salary" in x or "Benefits" in x))
    if comp_strong:
        next_p = comp_strong.parent.find_next_sibling("p")
        salary = next_p.get_text(strip=True) if next_p else None

    # --- Job Location ---
    job_location = None
    location_strong = soup.find("strong", string=lambda x: x and "Job Location" in x)
    if location_strong:
        next_p = location_strong.find_parent().find_next_sibling("p")
        job_location = next_p.get_text(strip=True) if next_p else None

    # --- Education / Experience ---
    edu_text = None
    edu = []
    qual_headers = soup.find_all("strong", string=lambda x: x and ("Education" in x or "Experience" in x or "Qualifications" in x))
    for header in qual_headers:
        next_ul = header.find_parent().find_next_sibling("ul")
        if next_ul:
            for li in next_ul.find_all("li"):
                edu.append(li.get_text(strip=True))
    edu_text = "; ".join(edu) if edu else None

    # --- Split Education/Experience into Qualification + Skills ---
    qualification, extracted_skills = split_edu_experience(edu_text)

    # --- Store results ---
    results.append({
        "Job Title": job_title,
        "Employer": employer,
        "Employer URL": employer_url,
        "Posted Date": posted_date,
        "Salary": salary,
        "Job Location": job_location,
        "Education/Experience": edu_text,
        "Qualification": qualification,
        "Skills": extracted_skills,
        "Source URL": jobs
    })


In [ ]:

# Convert to DataFrame
df = pd.DataFrame(results)

In [ ]:
df

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/131.0.0.0 Safari/537.36"
}

results = []

for jobs in job_url:
    response = requests.get(jobs, headers=headers, timeout=10)
    if response.status_code != 200:
        print(f"❌ Skipped {jobs}")
        continue

    soup = BeautifulSoup(response.content, "html.parser")

    # --- Job Title ---
    job_title_tag = soup.find("h1")
    job_title = job_title_tag.get_text(strip=True) if job_title_tag else None

    # --- Employer ---
    employer_tag = soup.find("p", string=lambda x: x and "Employer:" in x)
    if employer_tag:
        employer = employer_tag.get_text(strip=True).replace("Employer:", "").strip()
    else:
        employer_strong = soup.find("strong", string=lambda x: x and "Employer:" in x)
        employer = employer_strong.parent.get_text(strip=True).replace("Employer:", "").strip() if employer_strong else None

    # --- Employer URL ---
    employer_url_tag = soup.find("p", string=lambda x: x and "Employer URL:" in x)
    if employer_url_tag:
        employer_url = employer_url_tag.find("a")["href"] if employer_url_tag.find("a") else None
    else:
        employer_url_strong = soup.find("strong", string=lambda x: x and "Employer URL:" in x)
        employer_url = employer_url_strong.find_next("a")["href"] if employer_url_strong and employer_url_strong.find_next("a") else None

    # --- Posted Date ---
    posted_date = None
    for strong in soup.find_all("strong"):
        if "Posted to IASSIST on:" in strong.get_text():
            posted_date = strong.parent.get_text(strip=True).replace("Posted to IASSIST on:", "").strip()
            break

    # --- Salary ---
    salary = None
    for strong in soup.find_all("strong"):
        if "Salary" in strong.get_text():
            next_p = strong.parent.find_next_sibling("p")
            if next_p:
                salary = next_p.get_text(strip=True)
                break

    # --- Necessary Skills ---
    necessary_skills = []
    necessary_skills_label = soup.find(lambda tag: tag.name == "p" and "Necessary Skills:" in tag.get_text())
    if necessary_skills_label:
        ul = necessary_skills_label.find_next_sibling("ul")
        if ul:
            necessary_skills = [li.get_text(strip=True) for li in ul.find_all("li")]

    # --- Preferred Skills ---
    preferred_skills = []
    preferred_skills_label = soup.find(lambda tag: tag.name == "p" and "Preferred Skills:" in tag.get_text())
    if preferred_skills_label:
        ul = preferred_skills_label.find_next_sibling("ul")
        if ul:
            preferred_skills = [li.get_text(strip=True) for li in ul.find_all("li")]

    # --- Education and Experience ---
    education_and_experience = []
    edu_exp_label = soup.find(lambda tag: tag.name == "p" and "Education and Experience" in tag.get_text())
    if edu_exp_label:
        ul = edu_exp_label.find_next_sibling("ul")
        if ul:
            education_and_experience = [li.get_text(strip=True) for li in ul.find_all("li")]

    # --- Combine and store ---
    results.append({
        "Job Title": job_title,
        "Employer": employer,
        "Employer URL": employer_url,
        "Salary": salary,
        "Posted Date": posted_date,
        "Education and Experience": "; ".join(education_and_experience) if education_and_experience else None,
        "Necessary Skills": "; ".join(necessary_skills) if necessary_skills else None,
        "Preferred Skills": "; ".join(preferred_skills) if preferred_skills else None,
        "Source URL": jobs
    })



In [ ]:
# Example: convert to DataFrame
df = pd.DataFrame(results)

In [ ]:
job_url

In [ ]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/131.0.0.0 Safari/537.36"
}

results = []

for jobs in job_url:
    response = requests.get(jobs, headers=headers, timeout=10)
    if response.status_code != 200:
        print(f"❌ Skipped {jobs}")
        continue

    soup = BeautifulSoup(response.content, "html.parser")

    # Extract fields by label
    # --- Job Title ---
    job_title_tag = soup.find("h1")
    job_title = job_title_tag.get_text(strip=True) if job_title_tag else None

    # --- Employer ---
    employer_tag = soup.find("p", string=lambda x: x and "Employer:" in x)
    if employer_tag:
        employer = employer_tag.get_text(strip=True).replace("Employer:", "").strip()
    else:
        # fallback if fonts are nested
        employer_strong = soup.find("strong", string=lambda x: x and "Employer:" in x)
        employer = employer_strong.parent.get_text(strip=True).replace("Employer:", "").strip() if employer_strong else None

    # --- Employer URL ---
    employer_url_tag = soup.find("p", string=lambda x: x and "Employer URL:" in x)
    if employer_url_tag:
        employer_url = employer_url_tag.find("a")["href"] if employer_url_tag.find("a") else None
    else:
        # fallback
        employer_url_strong = soup.find("strong", string=lambda x: x and "Employer URL:" in x)
        employer_url = employer_url_strong.find_next("a")["href"] if employer_url_strong and employer_url_strong.find_next("a") else None

    # --- Necessary Skills ---
    skills_section = soup.find("p", string=lambda x: x and "Necessary Skills:" in x)
    skills = []
    if skills_section:
        ul = skills_section.find_next("ul")
        if ul:
            skills = [li.get_text(strip=True) for li in ul.find_all("li")]
    skills_text = "; ".join(skills) if skills else None

    # --- Salary ---
    salary_section = soup.find("p", string=lambda x: x and "Salary" in x)
    salary = None
    if salary_section:
        # salary info is usually in the next <p>
        salary_info = salary_section.find_next("p")
        if salary_info:
            salary = salary_info.get_text(strip=True)

    posted_date_tag = soup.find("p", string=lambda x: x and "Posted to IASSIST on:" in x)
    posted_date = None
    if posted_date_tag:
        posted_date = posted_date_tag.get_text(strip=True).replace("Posted to IASSIST on:", "").strip()

    # Store results
    results.append({
            "Job Title": job_title,
            "Employer": employer,
            "Employer URL": employer_url,
            "Salary": salary,
            "Skills": skills_text,
            "Date": posted_date,
            "Source URL": jobs
    })


In [ ]:
df = pd.DataFrame(results)

In [ ]:
df

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/131.0.0.0 Safari/537.36"
}

results = []

for jobs in job_url:
    response = requests.get(jobs, headers=headers, timeout=10)
    if response.status_code != 200:
        print(f"❌ Skipped {jobs}")
        continue

    soup = BeautifulSoup(response.content, "html.parser")

    # --- Job Title ---
    job_title_tag = soup.find("h1")
    job_title = job_title_tag.get_text(strip=True) if job_title_tag else None

    # --- Employer ---
    employer_tag = soup.find("p", string=lambda x: x and "Employer:" in x)
    if employer_tag:
        employer = employer_tag.get_text(strip=True).replace("Employer:", "").strip()
    else:
        employer_strong = soup.find("strong", string=lambda x: x and "Employer:" in x)
        employer = employer_strong.parent.get_text(strip=True).replace("Employer:", "").strip() if employer_strong else None

    # --- Employer URL ---
    employer_url_tag = soup.find("p", string=lambda x: x and "Employer URL:" in x)
    if employer_url_tag:
        employer_url = employer_url_tag.find("a")["href"] if employer_url_tag.find("a") else None
    else:
        employer_url_strong = soup.find("strong", string=lambda x: x and "Employer URL:" in x)
        employer_url = employer_url_strong.find_next("a")["href"] if employer_url_strong and employer_url_strong.find_next("a") else None

    # --- Necessary Skills ---
    skills = []
    skills_label = soup.find(lambda tag: tag.name=="p" and "Necessary Skills:" in tag.get_text())
    if skills_label:
        # Get the <ul> after the label
        ul = skills_label.find_next_sibling("ul")
        if ul:
            # Extract text from each <li>
            for li in ul.find_all("li"):
                skills.append(li.get_text(strip=True))
    skills_text = "; ".join(skills) if skills else None

    posted_date = None
    for strong in soup.find_all("strong"):
        if "Posted to IASSIST on:" in strong.get_text():
            posted_date = strong.parent.get_text(strip=True).replace("Posted to IASSIST on:", "").strip()
            break

# --- Salary ---
    salary = None
    for strong in soup.find_all("strong"):
        if "Salary" in strong.get_text():
        # salary is usually in the next <p> after this <strong>
            next_p = strong.parent.find_next_sibling("p")
            if next_p:
                salary = next_p.get_text(strip=True)
                break

    # Store results
    results.append({
        "Job Title": job_title,
        "Employer": employer,
        "Employer URL": employer_url,
        "Salary": salary,
        "Skills": skills_text,
        "Posted Date": posted_date,
        "Source URL": jobs
    })